In [ ]:
import re
from collections import deque
from functools import reduce
from pathlib import Path

import numpy as np
import pandas as pd


BASE = Path("/")
INPUT_DIR = BASE / "labeled_sorted"
OUTPUT_DIR = BASE / "feature_user"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "logon": "logon_labeled.csv",
    "device": "device_labeled.csv",
    "file": "file_labeled.csv",
    "email": "email_labeled.csv",
    "http": "http_labeled.csv",
}

WORK_START_HOUR = 8
WORK_END_HOUR = 17


def load_events(modality, columns):
    path = INPUT_DIR / FILES[modality]

    if not path.exists():
        raise FileNotFoundError(f"[{modality}] 입력 파일이 없습니다: {path}")

    df = pd.read_csv(path, usecols=columns, low_memory=False)

    if df.empty:
        raise ValueError(f"[{modality}] 0행 파일입니다.")

    if df["user"].isna().any() or df["date"].isna().any():
        raise ValueError(f"[{modality}] user 또는 date 결측값이 있습니다.")

    if not set(df["label"].dropna().unique()).issubset({0, 1}):
        raise ValueError(f"[{modality}] label은 0 또는 1이어야 합니다.")

    df["ts"] = pd.to_datetime(df["date"], errors="coerce")

    if df["ts"].isna().any():
        raise ValueError(f"[{modality}] date parsing 실패 행이 있습니다.")

    df = df.sort_values(["user", "ts"], kind="stable").reset_index(drop=True)
    df["day"] = df["ts"].dt.normalize()

    df["after_hours"] = (
        (df["ts"].dt.dayofweek >= 5) |
        (df["ts"].dt.hour < WORK_START_HOUR) |
        (df["ts"].dt.hour >= WORK_END_HOUR)
    )

    print(
        f"[{modality.upper()}] rows={len(df):,}, users={df['user'].nunique():,}, "
        f"event_positives={(df['label'] == 1).sum():,}"
    )
    return df


def add_daily_label(daily_df, event_df, prefix):
    labels = (
        event_df.groupby(["user", "day"], as_index=False)["label"]
        .max()
        .rename(columns={"label": f"{prefix}_event_label"})
    )
    return daily_df.merge(labels, on=["user", "day"], how="left")


def pair_durations(df, start_activity, end_activity):
    """Device Connect–Disconnect duration 계산용."""
    duration = pd.Series(np.nan, index=df.index, dtype=float)
    unmatched_start = 0
    unmatched_end = 0

    for _, group in df.groupby(["user", "pc"], sort=False):
        waiting = deque()

        for idx, row in group.iterrows():
            activity = str(row["activity"]).strip().lower()

            if activity == start_activity:
                waiting.append((idx, row["ts"]))

            elif activity == end_activity:
                if waiting:
                    start_idx, start_ts = waiting.popleft()
                    seconds = (row["ts"] - start_ts).total_seconds()

                    if seconds >= 0:
                        duration.loc[start_idx] = seconds / 60.0
                else:
                    unmatched_end += 1

        unmatched_start += len(waiting)

    return duration, unmatched_start, unmatched_end


def file_tree_depth(value):
    if pd.isna(value) or not str(value).strip():
        return 0

    depths = []
    for path in str(value).split(";"):
        parts = [p for p in re.split(r"[\\\\/]+", path.strip()) if p]
        depths.append(len(parts))

    return max(depths, default=0)


def file_extension(filename):
    if pd.isna(filename):
        return ""

    name = str(filename).replace("\\", "/").rsplit("/", 1)[-1]

    if "." not in name or name.startswith("."):
        return ""

    return name.rsplit(".", 1)[-1].lower()


def recipient_count(row):
    recipients = []

    for column in ["to", "cc", "bcc"]:
        value = row[column]

        if pd.notna(value) and str(value).strip():
            recipients.extend([x.strip() for x in str(value).split(";") if x.strip()])

    return len(recipients)


def has_external_recipient(row):
    recipients = []

    for column in ["to", "cc", "bcc"]:
        value = row[column]

        if pd.notna(value) and str(value).strip():
            recipients.extend([x.strip().lower() for x in str(value).split(";") if x.strip()])

    return int(any(not address.endswith("@dtaa.com") for address in recipients))


def parse_attachments(value):
    if pd.isna(value) or not str(value).strip():
        return 0, 0

    attachment_list = [x.strip() for x in str(value).split(";") if x.strip()]
    total_bytes = 0

    for attachment in attachment_list:
        matched = re.search(r"\((\d+)\)\s*$", attachment)

        if matched:
            total_bytes += int(matched.group(1))

    return len(attachment_list), total_bytes



# 1. Logon feature

logon = load_events("logon", ["date", "user", "pc", "activity", "label"])

logons = logon[logon["activity"].astype(str).str.lower() == "logon"].copy()
logoffs = logon[logon["activity"].astype(str).str.lower() == "logoff"].copy()

logon_daily = (
    logons.groupby(["user", "day"], as_index=False)
    .agg(
        logon_count=("activity", "size"),
        after_hours_logon_count=("after_hours", "sum"),
        logon_unique_pc_count=("pc", "nunique"),
    )
)

logoff_daily = (
    logoffs.groupby(["user", "day"], as_index=False)
    .agg(
        logoff_count=("activity", "size"),
        after_hours_logoff_count=("after_hours", "sum"),
    )
)

logon_daily = logon_daily.merge(logoff_daily, on=["user", "day"], how="outer")
logon_daily = add_daily_label(logon_daily, logon, "logon")



# 2. Device feature

device = load_events("device", ["date", "user", "pc", "file_tree", "activity", "label"])
device["usb_duration_min"], unmatched_connect, unmatched_disconnect = pair_durations(
    device, "connect", "disconnect"
)

connects = device[device["activity"].astype(str).str.lower() == "connect"].copy()
connects["file_tree_depth"] = connects["file_tree"].apply(file_tree_depth)

device_daily = (
    connects.groupby(["user", "day"], as_index=False)
    .agg(
        device_connect_count=("activity", "size"),
        after_hours_device_connect_count=("after_hours", "sum"),
        mean_usb_duration=("usb_duration_min", "mean"),
        mean_file_tree_depth=("file_tree_depth", "mean"),
    )
)

disconnect_daily = (
    device[device["activity"].astype(str).str.lower() == "disconnect"]
    .groupby(["user", "day"])
    .size()
    .rename("device_disconnect_count")
    .reset_index()
)

device_daily = device_daily.merge(disconnect_daily, on=["user", "day"], how="outer")
device_daily = add_daily_label(device_daily, device, "device")
print(f"[DEVICE] unmatched_connect={unmatched_connect:,}, unmatched_disconnect={unmatched_disconnect:,}")



# 3. File-copy feature

file_df = load_events(
    "file",
    ["date", "user", "pc", "filename", "activity",
     "to_removable_media", "from_removable_media", "label"]
)

file_df["extension"] = file_df["filename"].apply(file_extension)

file_daily = (
    file_df.groupby(["user", "day"], as_index=False)
    .agg(
        file_copy_count=("filename", "size"),
        after_hours_file_copy_count=("after_hours", "sum"),
        unique_extension_count=("extension", lambda x: x[x.ne("")].nunique()),
    )
)

file_daily = add_daily_label(file_daily, file_df, "file")



# 4. Email feature

email = load_events(
    "email",
    ["date", "user", "to", "cc", "bcc", "from", "activity",
     "size", "attachments", "label"]
)

email["recipient_count"] = email.apply(recipient_count, axis=1)
email["external_recipient_email"] = email.apply(has_external_recipient, axis=1)

attachment_info = email["attachments"].apply(parse_attachments)
email["attachment_count"] = attachment_info.apply(lambda x: x[0])
email["attachment_bytes"] = attachment_info.apply(lambda x: x[1])

email_daily = (
    email.groupby(["user", "day"], as_index=False)
    .agg(
        email_count=("activity", "size"),
        mean_recipient_count=("recipient_count", "mean"),
        external_recipient_email_count=("external_recipient_email", "sum"),
        mean_attachment_count=("attachment_count", "mean"),
        total_attachment_bytes=("attachment_bytes", "sum"),
    )
)

email_daily = add_daily_label(email_daily, email, "email")



# 5. HTTP feature

http = load_events("http", ["date", "user", "pc", "url", "label"])

def url_depth(url):
    if pd.isna(url) or not str(url).strip():
        return 0

    path = str(url).split("?", 1)[0].split("#", 1)[0]
    path = re.sub(r"^[a-zA-Z]+://[^/]+", "", path)

    return len([x for x in path.split("/") if x])

http["url_depth"] = http["url"].apply(url_depth)

http_daily = (
    http.groupby(["user", "day"], as_index=False)
    .agg(
        http_access_count=("url", "size"),
        mean_url_depth=("url_depth", "mean"),
    )
)

http_daily = add_daily_label(http_daily, http, "http")



# 6. user + date outer merge 및 사용자별 CSV 저장

daily_tables = [logon_daily, device_daily, file_daily, email_daily, http_daily]

features = reduce(
    lambda left, right: left.merge(right, on=["user", "day"], how="outer"),
    daily_tables
)

label_columns = [column for column in features.columns if column.endswith("_event_label")]
features["label"] = features[label_columns].fillna(0).max(axis=1).astype(int)
features = features.drop(columns=label_columns)

feature_columns = [column for column in features.columns if column not in ["user", "day", "label"]]
features[feature_columns] = features[feature_columns].fillna(0)

features = features.sort_values(["user", "day"], kind="stable").reset_index(drop=True)


features["is_weekend"] = (features["day"].dt.dayofweek >= 5).astype(int)
features["date"] = features["day"].dt.strftime("%Y-%m-%d")
features = features.drop(columns=["day"])


remaining_columns = [
    column for column in features.columns
    if column not in ["date", "user", "is_weekend"]
]

features = features[["date", "user", "is_weekend"] + remaining_columns]

if features.duplicated(["user", "date"]).any():
    raise RuntimeError("user + date 중복 행이 생성되었습니다.")

print(
    f"\n[FINAL] user-day rows={len(features):,}, users={features['user'].nunique():,}, "
    f"positive_days={(features['label'] == 1).sum():,}"
)

for user, user_features in features.groupby("user", sort=False):
    output_path = OUTPUT_DIR / f"{user}_daily_features.csv"

    user_features = user_features.sort_values("date", kind="stable")
    user_features.to_csv(output_path, index=False)

    saved = pd.read_csv(output_path)

    if len(saved) != len(user_features) or not saved["date"].is_monotonic_increasing:
        raise RuntimeError(f"[{user}] 저장 검증 실패")

print(f"CSV 저장 완료: {OUTPUT_DIR}")